# David's Preprocessing

Fixes from team preprocessing:
- **Keeps ID** for joining across notebooks
- **Keeps raw spending** (no log transform) so CLV math works
- **Keeps Dt_Customer** as a proper date column for campaign timing analysis
- Adds log-transformed spending as separate columns if needed

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/marketing_campaign.csv', sep='\t', encoding='ISO-8859-1')
print(f"Raw shape: {df.shape}")
df.head(3)

Raw shape: (2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0


In [2]:
# Drop constant columns
df = df.drop(columns=[col for col in df.columns if df[col].nunique() == 1])

# Impute missing income with median
df['Income'] = df['Income'].fillna(df['Income'].median())

print(f"After dropping constants: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

After dropping constants: (2240, 27)
Missing values: 0


In [3]:
# Convert Dt_Customer to datetime (KEEP IT)
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')

# Age from Year_Birth
df['Age'] = 2014 - df['Year_Birth']

# Customer tenure in days (from a fixed reference date, not today)
reference_date = df['Dt_Customer'].max()
df['Customer_Tenure_Days'] = (reference_date - df['Dt_Customer']).dt.days

print(f"Date range: {df['Dt_Customer'].min()} to {df['Dt_Customer'].max()}")
print(f"Age range: {df['Age'].min()} to {df['Age'].max()}")

Date range: 2012-07-30 00:00:00 to 2014-06-29 00:00:00
Age range: 18 to 121


In [4]:
# Remove outliers (same IQR method as team)
Q1_age, Q3_age = df['Age'].quantile(0.25), df['Age'].quantile(0.75)
IQR_age = Q3_age - Q1_age
df = df[(df['Age'] >= Q1_age - 1.5 * IQR_age) & (df['Age'] <= Q3_age + 1.5 * IQR_age)]

Q1_inc, Q3_inc = df['Income'].quantile(0.25), df['Income'].quantile(0.75)
IQR_inc = Q3_inc - Q1_inc
df = df[(df['Income'] >= Q1_inc - 1.5 * IQR_inc) & (df['Income'] <= Q3_inc + 1.5 * IQR_inc)]

print(f"After outlier removal: {df.shape}")

After outlier removal: (2229, 29)


In [5]:
# Consolidate Education
df['Education'] = df['Education'].str.strip().str.lower().replace({
    'basic': 'Undergraduate',
    '2n cycle': 'Undergraduate',
    'graduation': 'Graduate',
    'master': 'Postgraduate',
    'phd': 'PhD',
})

# Consolidate Marital_Status into Living_With
df['Marital_Status'] = df['Marital_Status'].str.strip().str.lower()
df['Living_With'] = df['Marital_Status'].replace({
    'married': 'Partner', 'together': 'Partner',
    'absurd': 'Alone', 'widow': 'Alone', 'yolo': 'Alone',
    'divorced': 'Alone', 'single': 'Alone', 'alone': 'Alone'
})

# Children + Family Size
df['Children_at_home'] = df['Kidhome'] + df['Teenhome']
df['Family_Size'] = df['Living_With'].map({'Alone': 1, 'Partner': 2}).fillna(1).astype(int) + df['Children_at_home']

print(f"Education: {df['Education'].unique()}")
print(f"Living_With: {df['Living_With'].unique()}")

Education: <StringArray>
['Graduate', 'PhD', 'Postgraduate', 'Undergraduate']
Length: 4, dtype: str
Living_With: <StringArray>
['Alone', 'Partner']
Length: 2, dtype: str


In [6]:
# RAW spending totals (no log transform)
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['Total_Spending'] = df[spend_cols].sum(axis=1)

# Total purchases
df['Total_Purchases'] = df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases']

print(f"Spending range: ${df['Total_Spending'].min():.0f} - ${df['Total_Spending'].max():.0f}")
print(f"Mean spending: ${df['Total_Spending'].mean():.0f}")

Spending range: $5 - $2525
Mean spending: $606


In [7]:
# Drop redundant columns but KEEP ID and Dt_Customer
df = df.drop(columns=['Kidhome', 'Teenhome', 'Marital_Status', 'Year_Birth'])

print(f"Final shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Final shape: (2229, 30)
Columns: ['ID', 'Education', 'Income', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Response', 'Age', 'Customer_Tenure_Days', 'Living_With', 'Children_at_home', 'Family_Size', 'Total_Spending', 'Total_Purchases']


,ID,Education,Income,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,AcceptedCmp2,Complain,Response,Age,Customer_Tenure_Days,Living_With,Children_at_home,Family_Size,Total_Spending,Total_Purchases
0,5524,Graduate,58138.0,2012-09-04,58,635,88,546,172,88,...,0,0,1,57,663,Alone,0,1,1617,22
1,2174,Graduate,46344.0,2014-03-08,38,11,1,6,2,1,...,0,0,0,60,113,Alone,2,3,27,4
2,4141,Graduate,71613.0,2013-08-21,26,426,49,127,111,21,...,0,0,0,49,312,Partner,0,2,776,20
3,6182,Graduate,26646.0,2014-02-10,26,11,4,20,10,3,...,0,0,0,30,139,Partner,1,3,53,6
4,5324,PhD,58293.0,2014-01-19,94,173,43,118,46,27,...,0,0,0,33,161,Partner,1,3,422,14


In [8]:
# Save
df.to_csv('../data/preprocessed/david_preprocessed.csv', index=False)
print(f"Saved {len(df)} rows to ../data/preprocessed/david_preprocessed.csv")

Saved 2229 rows to ../data/preprocessed/david_preprocessed.csv
